In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# IMPORT CONFIG (from your GitHub-synced config file) #
from config import FAERS_FILE_PATHS, BRONZE_BASE_PATH

# INITIALIZING SPARK SESSION #
spark = SparkSession.builder.appName("FAERS_PHARMA_ETL_PIPELINE").getOrCreate()

#---------------------------------------------------------------------------------------------------#
# PHASE 1 — EXTRACTION & RAW INGESTION #
# DATA READING #

# Reading FAERS files for 25Q4 #
dataframes = {}
for name, file in FAERS_FILE_PATHS.items():
    path = f"/Volumes/workspace/faers/faers_files/{file}"
    dataframes[name] = spark.read.csv(
        path,
        header=True,
        sep="$",
        inferSchema=True
    )

# Reading CT files for 25Q4 #
df_ct = spark.read.csv(
    "/Volumes/workspace/ct/ct_files/clinical_trials.csv",
    header=True,
    sep=",",
    inferSchema=True
)

#---------------------------------------------------------------------------------------------------#
# PHASE 2 — RAW PARQUET CONVERSION #

# SAVING FAERS FILES AS PARQUET #
for name, df in dataframes.items():
    bronze_raw_parquet_path = f"{BRONZE_BASE_PATH}/faers_parquet/raw/{name}"
    df.write.mode("overwrite").parquet(bronze_raw_parquet_path)
    print(f"{name.upper()} parquet file saved successfully")

# SAVING CT FILE AS PARQUET #
df_ct.write.mode("overwrite").parquet(
    f"{BRONZE_BASE_PATH}/ct_parquet/raw/ct"
)
print("CT parquet file saved successfully")

#---------------------------------------------------------------------------------------------------#